# Week2_ex2 - Straight Wire B-field Simulation

Simulates the B-field around a straight current-carrying wire with the EddyCurrent solver in Maxwell 3D, then compares the result against the theoretical Biot-Savart formula.


In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import matplotlib.pyplot as plt

In [ ]:
# 1. Ansys Electronics Desktop (AEDT) interface
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. Turn off autosave
DT.disable_autosave()

# 3. Create project and set solution type
sol_type = "EddyCurrent"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object for script recording
oDesign = M3D.odesign


In [ ]:
## Set up output directory for results ##
proj_name = "Week2_ex2"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

# 2. Save project
proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

# 3. Rename design
desi_name = "Week2_ex2"
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()


In [ ]:
# Function that finds the terminal faces of a coil
# Looks at the center coordinates of each face on the winding object, finds the face(s) with the largest
# absolute x position, and returns those two terminal faces
def find_terminal_face(winding_obj) :
    terminal_face = []

    # find maximum x position of winding object
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    # append terminal face to array
    for face in winding_obj.faces :
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001 :
            terminal_face.append(face)

    # sort
    ter_out, ter_in = sorted(terminal_face, key=lambda x : x.center[0], reverse=True)

    return ter_out, ter_in


In [ ]:
## Define design variables ##

line_length = 100
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_num_seg = 12
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"


In [ ]:
# 1. Copy & paste the setup code generated by script recording
oModule = oDesign.GetModule("AnalysisSetup")
oModule.InsertSetup("EddyCurrent", 
	[
		"NAME:Setup1",
		"Enabled:="		, True,
		[
			"NAME:MeshLink",
			"ImportMesh:="		, False
		],
		"MaximumPasses:="	, 10,
		"MinimumPasses:="	, 2,
		"MinimumConvergedPasses:=", 1,
		"PercentRefinement:="	, 15,
		"SolveFieldOnly:="	, False,
		"PercentError:="	, 1,
		"SolveMatrixAtLast:="	, True,
		"UseNonLinearIterNum:="	, False,
		"CacheSaveKind:="	, "Delta",
		"ConstantDelta:="	, "0s",
		"UseCacheFor:="		, ["Freq"],
		"UseIterativeSolver:="	, False,
		"RelativeResidual:="	, 1E-05,
		"NonLinearResidual:="	, 0.0001,
		"RelaxationFactor:="	, 1,
		"SmoothBHCurve:="	, True,
		"Frequency:="		, "10Hz",
		"HasSweepSetup:="	, False,
		"UseHighOrderShapeFunc:=", False,
		"ImportMeshForMuLink:="	, False,
		"LossAdaptiveCtrl:="	, "0.5",
		"UseMuLink:="		, False
	])

# 2. Save the setup name so I can call Analyze directly with the raw AEDT API later
# (on AEDT 2025 R2 the EddyCurrent solver got renamed to "AC Magnetic" internally,
#  and pyaedt's M3D.setups[-1] lookup doesn't recognize the new name -> KeyError)
setup_name = "Setup1"


In [ ]:
# 1. Draw the straight wire
point_tmp = []
point_tmp.append(["-line_length/2", "0mm", "0mm"])
point_tmp.append(["line_length/2", "0mm", "0mm"])
line = M3D.modeler.create_polyline(points=point_tmp, name="line", material="copper",
                                   xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

# 2. Create region and assign Radiation boundary condition
region_tmp = ["line_length/2", "-line_length/2", "line_length", "-line_length", "line_length", "-line_length"]
region = M3D.modeler.create_region(pad_value=region_tmp ,pad_type="Absolute Position")

# NOTE: M3D.assign_radiation() checks that solution_type == "EddyCurrent" exactly and
# raises AEDTRuntimeError("Excitation applicable only to Eddy Current.") otherwise. But
# AEDT 2025 R2 renamed this solver "AC Magnetic" internally, so self.solution_type reports
# "AC Magnetic" here -- the check fails even though this IS an eddy current solve. Bypassing
# the pyaedt wrapper and calling the raw AEDT boundary API directly sidesteps this check.
oModule = oDesign.GetModule("BoundarySetup")
face_ids = [region.top_face_z.id, region.bottom_face_z.id, region.top_face_y.id, region.bottom_face_y.id]
oModule.AssignRadiation(
	[
		"NAME:Radiation1",
		"Objects:=", [],
		"Faces:=", face_ids
	])

# 3. Create dummy object
box_origin = ["line_diameter", "line_length", "line_length"]
box_sizes = ["-2*line_diameter", "-2*line_length", "-2*line_length"]
dummy = M3D.modeler.create_box(origin=box_origin, sizes=box_sizes, name="dummy", material="vacuum")
dummy.subtract(tool_list=line, keep_originals=True)


# 4. Create a line to mark the region I want to extract data from
point_tmp = []
point_tmp.append(["0mm", "0mm", "0mm"])
point_tmp.append(["0mm", line_length, "0mm"])
field_line = M3D.modeler.create_polyline(points=point_tmp, name="field_line")


In [ ]:
## Mesh setting ##

oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:line_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["line"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/5",
	])

oModule.AssignLengthOp(
	[
		"NAME:dummy_mesh",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["dummy"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/5"
	])


In [ ]:
# 1. Set the terminal faces of the 3D object I'm using as a coil
ter_out, ter_in = find_terminal_face(line)  # call the function that finds the coil's terminal faces

M3D.assign_coil(assignment=ter_out, conductors_number=1, polarity="Negative", name="Out")
M3D.assign_coil(assignment=ter_in, conductors_number=1, polarity="Positive", name="In")


# 2. Create a Winding and add the coils I set up to it
coil = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=True, current="Current", 
                            resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, 
                            name="coil")

M3D.add_winding_coils(coil.name, coils=["Out", "In"])


In [ ]:
## analyze ##

# Using raw AEDT API here instead of pyaedt's setup.analyze() wrapper -
# same "AC Magnetic" renaming issue as above breaks the pyaedt setup lookup
oDesign.Analyze(setup_name)

In [ ]:
## Plot graph ## 

oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport("Calculator Expressions Plot 1", "Fields", "Rectangular Plot", "Setup1 : LastAdaptive", 
	[
		"Context:="		, "field_line",
		"PointCount:="		, 301
	], 
	[
		"Distance:="		, ["All"],
		"Freq:="		, ["All"],
		"Phase:="		, ["0deg"],
		"line_length:="		, ["Nominal"],
		"line_diameter:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "Distance",
		"Y Component:="		, ["Mag_B"]
	])


csv_name = f"{proj_name}_B_field_result.csv"
dir_csv = os.path.join(dir, csv_name)
oModule.ExportToFile("Calculator Expressions Plot 1", dir_csv, False)



In [ ]:
## Plot with Python ##


# 1) Read the CSV file
#    - includes a header row, assuming the first column is 'Distance (mm)' and the second is 'B (T)'
df = pd.read_csv(dir_csv)

# 2) Pull out the distance(mm) and field(T) columns from the CSV
distance_mm = df['Distance [mm]'].values
B_data      = df['Mag_B [mTesla]'].values

# 3) convert mm -> m
r_data = distance_mm * 1e-3

# 4) constants and wire parameters needed for the theoretical field calculation
mu0 = 4.0 * np.pi * 1e-7  # [H/m] permeability of free space
I   = Current               # [A]  current (from above)
R   = 0.005               # [m]  wire radius (e.g. 1 mm)

# 5) formulas for the field inside/outside the wire
#    - inside (r <= R): B_inner = (mu0 * r * I) / (2π R^2)
#    - outside (r >  R): B_outer = (mu0 * I)     / (2π r)
def B_theory(r_array):
    return np.where(
        r_array <= R,
        mu0 * r_array * I / (2.0 * np.pi * R**2),  # inside
        mu0 * I           / (2.0 * np.pi * r_array) # outside
    )

# calculate theoretical value
B_calc = B_theory(r_data)
B_calc = B_calc*1e3    # convert to [mT]

# 6) plot
plt.figure(figsize=(8,6))
plt.plot(distance_mm, B_data, 'ro-',  label='Ansys simulation')
plt.plot(distance_mm, B_calc, 'b--', label='formula')
plt.xlabel('distance (mm)')
plt.ylabel('B (mT)')
plt.title(' ')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
## Save project ##

M3D.save_project()